# 02_fwmodelling_and_data_visualization

GUI for running FD modelling and visualizing `Hxshot.rss`/`Hzshot.rss` outputs.
Also fits the global FDTD–analytic calibration `C(f)` used by Steps 05/06
(homogeneous `rho_min` or 1D lateral average of the true model; last successful
run overwrites the active `C` in `setup_metadata.json`).


In [ ]:
from pathlib import Path
import json
import os
import re
import signal
import subprocess
import threading
import time
import traceback
import sys

# Project root: start scripts run from repo root, so cwd is the workshop directory
ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.modules.workshop_config import load_config
CONFIG = load_config()
WORKSPACE = CONFIG.workspace

import numpy as np

try:
    import ipywidgets as ipw
    import plotly.graph_objects as go
except Exception as exc:
    raise RuntimeError(
        'Missing GUI dependencies. Install with: pip install voila ipywidgets plotly numpy ipykernel matplotlib scipy segyio'
    ) from exc

from scripts.modules import rockem_bridge
from scripts.modules.fd_visualization import compute_gains_for_fd_outputs, save_amp_phase_npz
from scripts.modules.fdtd_analytic_calibration import (
    CALIBRATION_CFG_NAME,
    DEFAULT_SOURCE_FIELD,
    METHOD_HOMOGENEOUS,
    METHOD_LATERAL_AVERAGE,
    SOURCE_FIELDS,
    apply_calibration_to_gains,
    calibration_consistency_warning,
    calibration_run_dir,
    compute_calibration_from_fdtd_outputs,
    load_global_calibration,
    prepare_homogeneous_calibration_run,
    prepare_lateral_average_calibration_run,
    save_calibration_to_metadata,
)
from scripts.modules.setup_defaults import (
    default_frequencies as _default_frequencies,
    default_f_min_hz as _default_f_min_hz_meta,
    default_n_periods_extract as _default_n_periods_extract,
    format_freq_list,
    status_note_if_meta_missing,
)

FWD_2D_DIR = CONFIG.fwd_2d_dir
DATA_DIR = FWD_2D_DIR / 'Data'
HX_PATH = DATA_DIR / 'Hxshot.rss'
HZ_PATH = DATA_DIR / 'Hzshot.rss'
WAV2D_PATH = FWD_2D_DIR / 'wav2d.rss'
MPIQUEUE_LOG = FWD_2D_DIR / 'mpiqueue.log'
RUN_SCRIPT = FWD_2D_DIR / 'runmod.sh'
SETUP_META = FWD_2D_DIR / 'setup_metadata.json'
OUT_DIR = DATA_DIR / 'processed'
OUT_NPZ = OUT_DIR / 'amp_phase_results.npz'
CAL_DIR = calibration_run_dir(FWD_2D_DIR, METHOD_HOMOGENEOUS)

# --- acquisition matrix -----------------------------------------------------
# Step 01 can now emit one dataset per (frequency, source component) pair, listed
# in `manifest.json`. Everything below this point works on ONE dataset at a time,
# through the paths bound above; `_select_dataset` rebinds them so the whole
# notebook follows the dropdown without threading a dataset argument through
# ~58 call sites. `iter_datasets` falls back to treating the forward directory
# itself as one dataset, so a workspace built before the matrix still loads.
FWD_ROOT = CONFIG.fwd_2d_dir


def available_datasets():
    from scripts.modules.headless import iter_datasets
    try:
        return iter_datasets(FWD_ROOT)
    except Exception:
        return []


def _select_dataset(name=None):
    """Point every path constant at one dataset of the acquisition matrix."""
    global FWD_2D_DIR, DATA_DIR, HX_PATH, HZ_PATH, WAV2D_PATH
    global MPIQUEUE_LOG, RUN_SCRIPT, SETUP_META, OUT_DIR, OUT_NPZ, CAL_DIR
    ds = available_datasets()
    chosen = None
    if ds:
        chosen = next((d for d in ds if d['name'] == name), ds[0])
    FWD_2D_DIR = Path(chosen['run_dir']) if chosen else FWD_ROOT
    DATA_DIR = FWD_2D_DIR / 'Data'
    HX_PATH = DATA_DIR / 'Hxshot.rss'
    HZ_PATH = DATA_DIR / 'Hzshot.rss'
    WAV2D_PATH = FWD_2D_DIR / 'wav2d.rss'
    MPIQUEUE_LOG = FWD_2D_DIR / 'mpiqueue.log'
    RUN_SCRIPT = FWD_2D_DIR / 'runmod.sh'
    SETUP_META = FWD_2D_DIR / 'setup_metadata.json'
    OUT_DIR = DATA_DIR / 'processed'
    OUT_NPZ = OUT_DIR / 'amp_phase_results.npz'
    CAL_DIR = calibration_run_dir(FWD_2D_DIR, METHOD_HOMOGENEOUS)
    return chosen
VOILA_PID_FILE = ROOT / '.voila_fwmodelling_server.pid'
MAX_CPUS = max(2, os.cpu_count() or 2)
# See scripts.modules.rockem_bridge: the validated checkout, not the stale
# ~/software/rockem-suite one. 2D now defaults to the EXPLICIT engine
# (mpiEmmodTE2d) - ADI TE2D fails the suite's own layered-model validation.
MPI_EMMOD_BIN_2D = str(rockem_bridge.binary_path(CONFIG.forward_engine_te2d()))
MPI_EMMOD_BIN_3D = str(rockem_bridge.ROCKEM_SUITE_ROOT / 'bin' / 'mpiEmmodADI3d')

state = {
    'process': None,
    'result': None,
    'calibration': None,
    'last_messages': [],
    'refresh_loop': False,
    'monitor_thread': None,
    'run_started': False,
}


def push_message(msg):
    state['last_messages'].append(msg)
    if len(state['last_messages']) > 12:
        state['last_messages'] = state['last_messages'][-12:]
    status_area.value = '\n'.join(state['last_messages'])


def bind_button_with_feedback(button, handler, action_label, success_message=None):
    def _wrapped(_):
        before = len(state.get('last_messages', []))
        push_message(f'{action_label}...')
        try:
            handler(_)
            after = len(state.get('last_messages', []))
            if after <= before + 1:
                push_message(success_message or f'{action_label} completed successfully.')
        except Exception as exc:
            push_message(f'{action_label} failed: {exc}')
            raise

    button.on_click(_wrapped)


def read_tail(path, n_lines=30):
    path = Path(path)
    if not path.exists():
        return f'{path} not found.'
    lines = path.read_text(errors='replace').splitlines()
    tail = lines[-int(n_lines):] if lines else []
    return '\n'.join(tail) if tail else '(empty)'


def progress_dir():
    """Directory the progress panels read from.

    During a batch this is the dataset CURRENTLY RUNNING, not the one selected
    in the view dropdown - otherwise the panels sit empty while a different
    dataset is being modelled, which is exactly how they used to behave.
    """
    d = state.get('active_run_dir')
    return Path(d) if d else FWD_2D_DIR


def discover_job_logs():
    root = progress_dir()
    logs = []
    for p in sorted(root.glob('*.log')):
        if p.name != 'mpiqueue.log':
            logs.append(p)
    for p in sorted(root.glob('log.txt-*')):
        logs.append(p)
    return logs


def _last_nonempty_line(path):
    path = Path(path)
    if not path.exists():
        return ''
    lines = path.read_text(errors='replace').splitlines()
    for line in reversed(lines):
        if line.strip():
            return line.strip()
    return ''


def _extract_percent_from_line(line):
    if not line:
        return np.nan
    m = re.search(r'(\d+(?:\.\d+)?)\s*%', line)
    if m:
        try:
            return float(m.group(1))
        except Exception:
            return np.nan
    if 'complete' in line.lower() or 'finished' in line.lower():
        return 100.0
    return np.nan


def summarize_job_progress(logs):
    rows = []
    for lp in logs:
        last = _last_nonempty_line(lp)
        pct = _extract_percent_from_line(last)
        rows.append({'name': lp.name, 'percent': pct, 'last_line': last})
    return rows


def all_jobs_complete(rows):
    if not rows:
        return False
    for row in rows:
        pct = row['percent']
        if np.isnan(pct) or pct < 100.0:
            return False
    return True


def default_frequencies():
    return _default_frequencies(path=SETUP_META)


def resolve_forward_command_from_metadata():
    meta = {}
    if SETUP_META.exists():
        try:
            meta = json.loads(SETUP_META.read_text())
        except Exception:
            meta = {}

    data_dim = int(meta.get('forward_data_dim', 2))
    forward_cfg = str(meta.get('forward_cfg', 'mod.cfg'))
    default_engine = 'mpiEmmodADI3d' if data_dim == 3 else CONFIG.forward_engine_te2d()
    engine_name = str(meta.get('forward_engine', default_engine))
    engine_path = str(rockem_bridge.ROCKEM_SUITE_ROOT / 'bin' / engine_name)
    fallback_note = None if SETUP_META.exists() else 'No setup metadata found; using 2D explicit-engine defaults.'

    return {
        'forward_data_dim': data_dim,
        'forward_cfg': forward_cfg,
        'forward_engine': engine_name,
        'engine_path': engine_path,
        'fallback_note': fallback_note,
    }


def parse_freqs(text):
    parts = [p.strip() for p in str(text).split(',') if p.strip()]
    if not parts:
        raise ValueError('Frequency list cannot be empty.')
    vals = np.asarray([float(p) for p in parts], dtype=float)
    if np.any(vals <= 0):
        raise ValueError('Frequencies must be positive.')
    return vals


def clear_stale_run_logs(announce=False, run_dir=None):
    removed = []
    root = Path(run_dir) if run_dir else progress_dir()
    mq = root / 'mpiqueue.log'
    if mq.exists():
        mq.unlink()
        removed.append(mq.name)
    for lp in ([p for p in sorted(root.glob('*.log')) if p.name != 'mpiqueue.log']
               + sorted(root.glob('log.txt-*'))):
        try:
            lp.unlink()
            removed.append(lp.name)
        except Exception:
            pass
    if announce and removed:
        push_message('Removed stale logs: ' + ', '.join(removed))
    return removed


def refresh_run_status(_=None):
    proc = state.get('process')
    process_active = False
    batch = state.get('batch') or {}
    where = f" [{batch.get('i', '?')}/{batch.get('n', '?')} {batch.get('name', '')}]" if batch else ''
    if proc is not None:
        rc = proc.poll()
        if rc is None:
            process_active = True
            run_status.value = f'Run process active (pid={proc.pid}){where}'
        else:
            run_status.value = f'Last run process exited with code {rc}{where}'
            state['process'] = None
    elif state.get('batch_running'):
        run_status.value = f'Batch running{where}'
        process_active = True
    else:
        run_status.value = 'No active run process in this session.'

    if (not state.get('run_started')) and (not process_active):
        job_progress_out.value = 'Run not started in this session yet.'
        worker_logs_out.value = 'Run not started in this session yet.'
        mpiqueue_out.value = 'Run not started in this session yet.'
        return

    mpiqueue_out.value = read_tail(progress_dir() / 'mpiqueue.log', n_lines=40)
    logs = discover_job_logs()
    if logs:
        rows = summarize_job_progress(logs)
        progress_lines = []
        for row in rows:
            pct = row['percent']
            pct_text = 'n/a' if np.isnan(pct) else f'{pct:.1f}%'
            progress_lines.append(f"{row['name']}: {pct_text} | {row['last_line']}")
        job_progress_out.value = '\n'.join(progress_lines)

        logs_text = []
        for lp in logs[:4]:
            logs_text.append(f'--- {lp.name} ---')
            logs_text.append(read_tail(lp, n_lines=15))
        worker_logs_out.value = '\n'.join(logs_text)

        if state.get('refresh_loop') and (not process_active) and all_jobs_complete(rows):
            state['refresh_loop'] = False
            push_message('All jobs completed. Auto-refresh stopped.')
    else:
        job_progress_out.value = 'No worker log files found yet.'
        worker_logs_out.value = 'No worker log files found yet.'


def start_auto_refresh():
    if state.get('refresh_loop'):
        return
    state['refresh_loop'] = True

    def _poll_loop():
        while state.get('refresh_loop'):
            try:
                refresh_run_status()
            except Exception:
                pass
            time.sleep(2.0)

    th = threading.Thread(target=_poll_loop, daemon=True)
    state['monitor_thread'] = th
    th.start()


def stop_auto_refresh():
    state['refresh_loop'] = False


def on_run_model(_):
    try:
        if not RUN_SCRIPT.exists():
            raise FileNotFoundError(f'Missing run script: {RUN_SCRIPT}')
        if state.get('process') is not None and state['process'].poll() is None:
            raise RuntimeError('A run process is already active in this GUI session.')

        clear_stale_run_logs(announce=True)

        DATA_DIR.mkdir(parents=True, exist_ok=True)

        nproc = max(2, min(int(getattr(nproc_input, 'value', 4)), MAX_CPUS))
        cmd_cfg = resolve_forward_command_from_metadata()
        engine_path = cmd_cfg['engine_path']
        cfg_name = cmd_cfg['forward_cfg']

        if cmd_cfg.get('fallback_note'):
            push_message(cmd_cfg['fallback_note'])

        cfg_path = FWD_2D_DIR / cfg_name
        if not cfg_path.exists():
            raise FileNotFoundError(f'Forward config file not found: {cfg_path}')
        if not os.path.isfile(engine_path):
            raise FileNotFoundError(f'MPI FD binary not found: {engine_path}')

        state['run_started'] = True
        job_progress_out.value = 'Waiting for new job logs...'
        worker_logs_out.value = 'Waiting for new worker logs...'
        mpiqueue_out.value = 'Waiting for new mpiqueue.log...'

        proc = subprocess.Popen(
            [CONFIG.mpirun, '-np', str(nproc), engine_path, cfg_name],
            cwd=str(FWD_2D_DIR),
        )
        state['process'] = proc
        push_message(
            f"Started {cmd_cfg['forward_engine']} (dim={cmd_cfg['forward_data_dim']}) "
            f"with {cfg_name} via mpirun -np {nproc} (pid={proc.pid})."
        )
        refresh_run_status()
        start_auto_refresh()
    except Exception as exc:
        push_message(f'Run start failed: {exc}')
        push_message(traceback.format_exc())


def on_stop_model(_):
    try:
        # Halt the whole sequential batch, not just the dataset currently
        # running - otherwise Stop simply lets the next one start.
        if state.get('batch_running'):
            state['batch_stop'] = True
            push_message('Batch stop requested - finishing the current dataset.')
        proc = state.get('process')
        if proc is None or proc.poll() is not None:
            push_message('No active run process to stop in this session.')
            stop_auto_refresh()
            return
        os.kill(proc.pid, signal.SIGINT)
        push_message(f'Sent SIGINT to run process pid={proc.pid}.')
        stop_auto_refresh()
        refresh_run_status()
    except Exception as exc:
        push_message(f'Stop run failed: {exc}')
        push_message(traceback.format_exc())


def on_load_outputs(_):
    try:
        missing = [str(p) for p in [HX_PATH, HZ_PATH] if not p.exists()]
        if missing:
            raise RuntimeError('Missing output files: ' + ', '.join(missing))
        load_info.value = f'Found outputs: Hx={HX_PATH.name}, Hz={HZ_PATH.name}'
        push_message('FD output files detected.')
    except Exception as exc:
        push_message(f'Load outputs failed: {exc}')
        push_message(traceback.format_exc())


def default_f_min_hz():
    if SETUP_META.exists():
        try:
            meta = json.loads(SETUP_META.read_text())
            v = meta.get('f_min_hz')
            if v:
                return float(v)
        except Exception:
            pass
    return float(min(default_frequencies()))


def on_compute(_):
    """Extract channel gains for EVERY dataset of the matrix.

    Step 01 decided which frequencies and sources exist; nothing here re-asks.
    Each dataset is extracted at ITS OWN tones and its own `n_periods_extract`
    (a per-frequency dataset contains only the tone it was designed for, so
    asking it for the whole band would read noise at the others). Results are
    kept per dataset in `state['results_by_dataset']`; the plots below show the
    one named in the `view dataset` selector.
    """
    try:
        ds = available_datasets() or [{'name': '(single)', 'run_dir': str(FWD_ROOT)}]
        by_ds = {}
        for i, d in enumerate(ds, 1):
            _select_dataset(d['name'])
            # Frequencies, f_min and n_periods_extract come from THIS dataset's
            # own setup_metadata.json. They are not user inputs: Step 01 wrote
            # them, and a per-frequency dataset only contains the tone it was
            # designed for, so anything else would be reading noise.
            meta = d.get('meta') or json.loads((Path(d['run_dir']) / 'setup_metadata.json').read_text())
            freqs = np.asarray(meta['flist_hz'], dtype=float)
            f_min_hz = float(meta['f_min_hz'])
            npx = float(meta['n_periods_extract'])
            if not WAV2D_PATH.exists():
                raise FileNotFoundError(f'Wavelet file not found: {WAV2D_PATH}')
            by_ds[d['name']] = compute_gains_for_fd_outputs(
                HX_PATH, HZ_PATH, WAV2D_PATH,
                freqs=freqs, f_min_hz=f_min_hz, n_periods_extract=npx,
            )
            push_message(f'[{i}/{len(ds)}] extracted gains for {d["name"]} '
                         f'({", ".join(f"{f:g}" for f in freqs)} Hz)')
        state['results_by_dataset'] = by_ds
        view_dataset.options = [(k, k) for k in by_ds]
        if view_dataset.value not in by_ds:
            view_dataset.value = next(iter(by_ds))
        result = by_ds[view_dataset.value]
        state['result'] = result
        nfreq = sum(len(r['Hx']['freqs']) for r in by_ds.values())
        ntr = result['geometry']['ntrace']

        # Stability backstop: a diverging/unstable run shows up here as a
        # non-finite channel gain - flag it rather than silently plotting NaN.
        n_bad = int(np.count_nonzero(~np.isfinite(result['Hx']['gain'])) + np.count_nonzero(~np.isfinite(result['Hz']['gain'])))
        if n_bad > 0:
            push_message(f'WARNING: {n_bad} non-finite channel-gain values (Hx/Hz) - check the FD run for instability/divergence.')

        comp_freq.options = [(f'{f:g} Hz', k) for k, f in enumerate(result['Hx']['freqs'])]
        comp_freq.value = 0
        trace_idx.max = max(0, ntr - 1)
        tx_ids = np.unique(result['geometry']['tx_idx_per_trace'])
        tx_select.options = [(f'tx#{int(i)}', int(i)) for i in tx_ids]
        if len(tx_ids) > 0:
            tx_select.value = int(tx_ids[0])

        rx_local = np.asarray(result['geometry'].get('rx_local_idx_per_trace', []), dtype=int)
        if rx_local.size > 0:
            rx_local_select.max = int(np.max(rx_local))
            rx_local_select.value = min(rx_local_select.value, rx_local_select.max)
        else:
            rx_local_select.max = 0
            rx_local_select.value = 0

        compute_info.value = (f'Computed channel-gain amp/phase for {len(by_ds)} dataset(s), '
                             f'{nfreq} frequency-dataset pair(s), {ntr} traces each.')
        push_message('Computed FD steady-state channel-gain amp/phase results (trace/wavelet phasor ratio).')
        update_plot()
    except Exception as exc:
        push_message(f'Compute failed: {exc}')
        push_message(traceback.format_exc())


def traces_for_tx(result, tx_id):
    tx_idx = np.asarray(result['geometry']['tx_idx_per_trace'])
    return np.where(tx_idx == int(tx_id))[0]


def update_plot(_=None):
    result = state.get('result')
    if result is None:
        return

    comp = component_select.value
    metric = metric_select.value
    tx_id = tx_select.value
    if tx_id is None:
        return

    freq_i = comp_freq.value if comp_freq.value is not None else 0
    tr_idx = traces_for_tx(result, tx_id)
    if tr_idx.size == 0:
        return

    geo = result['geometry']
    rx_local_ids = np.asarray(geo.get('rx_local_idx_per_trace', geo['rx_idx_per_trace']))
    comp_data = result[comp]
    fig = go.Figure()

    if metric == 'amp_vs_rx':
        x = rx_local_ids[tr_idx]
        order = np.argsort(x)
        x = x[order]
        y = comp_data['amp_mean'][freq_i, tr_idx][order]
        title = f'{comp} amplitude vs local rx (tx#{tx_id}, f={comp_data["freqs"][freq_i]:g} Hz)'
        ytitle = 'Amplitude'
        xtitle = 'Local receiver index'
    elif metric == 'phase_vs_rx_deg':
        x = rx_local_ids[tr_idx]
        order = np.argsort(x)
        x = x[order]
        y = np.degrees(comp_data['phi_mean_rad'][freq_i, tr_idx][order])
        title = f'{comp} phase vs local rx (tx#{tx_id}, f={comp_data["freqs"][freq_i]:g} Hz)'
        ytitle = 'Phase (deg)'
        xtitle = 'Local receiver index'
    elif metric == 'amp_vs_tx':
        fixed_rx = int(rx_local_select.value)
        tx_all = np.asarray(geo['tx_idx_per_trace'])
        mask = rx_local_ids == fixed_rx
        tx_vals = np.unique(tx_all[mask])
        x = []
        y = []
        for tx in tx_vals:
            tr = np.where((tx_all == tx) & mask)[0]
            if tr.size > 0:
                x.append(int(tx))
                y.append(float(np.nanmean(comp_data['amp_mean'][freq_i, tr])))
        x = np.asarray(x)
        y = np.asarray(y)
        title = f'{comp} amplitude vs tx (local rx={fixed_rx}, f={comp_data["freqs"][freq_i]:g} Hz)'
        ytitle = 'Amplitude'
        xtitle = 'Transmitter index'
    elif metric == 'phase_vs_tx_deg':
        fixed_rx = int(rx_local_select.value)
        tx_all = np.asarray(geo['tx_idx_per_trace'])
        mask = rx_local_ids == fixed_rx
        tx_vals = np.unique(tx_all[mask])
        x = []
        y = []
        for tx in tx_vals:
            tr = np.where((tx_all == tx) & mask)[0]
            if tr.size > 0:
                phi = comp_data['phi_mean_rad'][freq_i, tr]
                phi_mean = np.angle(np.nanmean(np.exp(1j * phi)))
                x.append(int(tx))
                y.append(float(np.degrees(phi_mean)))
        x = np.asarray(x)
        y = np.asarray(y)
        title = f'{comp} phase vs tx (local rx={fixed_rx}, f={comp_data["freqs"][freq_i]:g} Hz)'
        ytitle = 'Phase (deg)'
        xtitle = 'Transmitter index'
    elif metric == 'amp_vs_freq':
        global_trace = int(min(trace_idx.value, comp_data['ntrace'] - 1))
        y = comp_data['amp_mean'][:, global_trace]
        x = comp_data['freqs']
        title = f'{comp} amplitude vs frequency (trace#{global_trace})'
        ytitle = 'Amplitude'
        xtitle = 'Frequency (Hz)'
    else:
        global_trace = int(min(trace_idx.value, comp_data['ntrace'] - 1))
        y = np.degrees(comp_data['phi_mean_rad'][:, global_trace])
        x = comp_data['freqs']
        title = f'{comp} phase vs frequency (trace#{global_trace})'
        ytitle = 'Phase (deg)'
        xtitle = 'Frequency (Hz)'

    fig.add_trace(go.Scatter(x=x, y=y, mode='lines+markers', name=comp))
    fig.update_layout(title=title, xaxis_title=xtitle, yaxis_title=ytitle, height=420)
    with plot_out:
        plot_out.clear_output(wait=True)
        fig.show()


def on_save_results(_):
    """Save the processed results of EVERY dataset, each beside its own data."""
    try:
        by_ds = state.get('results_by_dataset')
        if not by_ds:
            raise RuntimeError('Compute results first.')
        for name, res in by_ds.items():
            _select_dataset(name)
            save_amp_phase_npz(OUT_NPZ, res)
            push_message(f'Saved {name} -> {OUT_NPZ}')
        _select_dataset(None)
        push_message(f'Saved processed results for {len(by_ds)} dataset(s).')
    except Exception as exc:
        push_message(f'Save failed: {exc}')
        push_message(traceback.format_exc())


def on_quit_gui(_):
    try:
        stop_auto_refresh()

        proc = state.get('process')
        if proc is not None and proc.poll() is None:
            os.kill(proc.pid, signal.SIGINT)
            push_message(f'Sent SIGINT to run process pid={proc.pid}.')

        pid = None
        if VOILA_PID_FILE.exists():
            pid_text = VOILA_PID_FILE.read_text().strip()
            if pid_text:
                pid = int(pid_text)

        if pid is not None:
            if pid != os.getpid():
                os.kill(pid, signal.SIGINT)
                push_message(f'Sent SIGINT to Voila server PID {pid}.')
            else:
                push_message('Voila PID matches current kernel process; skipping direct PID kill.')
        else:
            push_message('Voila PID file not found; proceeding with kernel shutdown.')
    except Exception as exc:
        push_message(f'Quit warning: {exc}. Proceeding with kernel shutdown.')
    finally:
        try:
            from IPython import get_ipython

            ip = get_ipython()
            if ip and getattr(ip, 'kernel', None):
                ip.kernel.do_shutdown(restart=False)
        except Exception:
            pass


# The single-dataset calibration path is GONE. Step 01 decides which
# frequencies and sources exist; calibration then applies to all of them, with
# one Earth model, in one action. A per-dataset button meant the operator
# carried the acquisition matrix in their head and could leave the workspace
# half-calibrated, or mixed across Earth models - exactly the state
# `calibration_consistency_warning` had to be invented to catch.


def on_calibrate_all(_):
    """Calibrate EVERY dataset of the matrix, sequentially, with one method.

    Choosing N frequencies and both sources in Step 01 must not become 2N rounds
    of picking a dataset, picking a source and pressing a button. Each dataset is
    calibrated with the source it was MODELLED with, which is exactly what the
    1D tensor inversion reads back per source.

    One method for all of them on purpose - mixing methods across sources is the
    mistake `calibration_consistency_warning` exists to catch, and a batch loop
    is the easiest possible way to commit it.

    The active dataset selection is restored afterwards so the rest of the
    notebook is where the user left it.
    """
    from scripts.modules.headless import run_calibration_matrix

    ds = available_datasets()
    if not ds:
        push_message('No datasets to calibrate - run Step 01 first.')
        return
    method = str(cal_all_method.value)
    nproc = max(2, min(int(getattr(nproc_input, 'value', 4)), MAX_CPUS))

    def _progress(i, n, d, entry):
        ok = 'ok' if entry.get('ok') else f'FAILED ({entry.get("error", "")})'
        push_message(f'[{i}/{n}] {d["name"]} [{entry["source_field"]}]: {ok}')

    push_message(f'Calibrating {len(ds)} dataset(s) with {method} - this blocks the GUI.')
    try:
        results = run_calibration_matrix(
            FWD_ROOT, method=method, nproc=nproc, datasets=ds,
            verbose=False, on_progress=_progress,
        )
    except Exception as exc:
        push_message(f'Batch calibration failed: {exc}')
        return
    finally:
        _select_dataset(None)

    n_ok = sum(1 for r in results if r.get('ok'))
    bad = [f'{r["name"]} [{r["source_field"]}]' for r in results if not r.get('ok')]
    warned = [r for r in results if r.get('warning')]
    for r in warned:
        push_message(f'WARNING {r["name"]} [{r["source_field"]}]: {r["warning"]}')
    cal_info.value = (
        f'<b>Batch calibration: {n_ok}/{len(results)} dataset-source pairs '
        f'with {method}.</b>'
        + (f'<br><b style="color:#b00">Failed: {", ".join(bad)}</b>' if bad else '')
        + (f'<br><b style="color:#b00">Earth-model clash left in '
           f'{len(warned)} dataset(s): {warned[0]["warning"]}</b>' if warned else '')
        + '<br>Every dataset carries its own C in its own setup_metadata.json.'
    )
    push_message(f'Batch calibration done: {n_ok}/{len(results)} succeeded.')
    # Keep the IN-MEMORY calibrations: they carry `fdtd_result`/`analytic_*`,
    # which the on-disk metadata deliberately does not, and the QC plot needs
    # them to compare FDTD against C x analytic.
    state['calibrations_by_dataset'] = {
        f'{r["name"]} [{r["source_field"]}]': r['cal']
        for r in results if r.get('ok') and r.get('cal') is not None
    }
    state['calibration'] = next(iter(state['calibrations_by_dataset'].values()), None)
    if state['calibration'] is not None:
        _f = np.asarray(state['calibration']['freqs_hz'], dtype=float)
        cal_freq_select.options = [(f'{f:g} Hz', k) for k, f in enumerate(_f)]
        cal_freq_select.value = 0
    update_calibration_plot()


def update_calibration_plot(_=None):
    """Calibration QC for the dataset in `view dataset`, at the selected tone.

    Uses the IN-MEMORY calibrations the batch returns: `fdtd_result` and
    `analytic_*` are stripped by `_json_safe_calibration_payload` before the
    metadata is written, so a calibration reloaded from disk can never draw
    this plot - it used to return here silently and show nothing at all.
    """
    cals = state.get('calibrations_by_dataset') or {}
    cal = None
    if cals:
        cal = next((c for k, c in cals.items() if k.startswith(str(view_dataset.value))), None)
        cal = cal or next(iter(cals.values()))
    cal = cal or state.get('calibration')
    if cal is None:
        return
    fdtd = cal.get('fdtd_result')
    if fdtd is None:
        return
    c_arr = np.asarray(cal['C_hxhz_shared'], dtype=complex)
    hx_a = np.asarray(cal['analytic_hx'], dtype=complex)
    hz_a = np.asarray(cal['analytic_hz'], dtype=complex)
    hx_cal, hz_cal = apply_calibration_to_gains(hx_a, hz_a, c_arr)
    hx_f = np.asarray(fdtd['Hx']['gain'], dtype=complex)
    hz_f = np.asarray(fdtd['Hz']['gain'], dtype=complex)
    freqs = np.asarray(cal['freqs_hz'], dtype=float)
    fidx = int(cal_freq_select.value) if cal_freq_select.value is not None else 0
    fidx = max(0, min(fidx, freqs.size - 1))
    comp = cal_comp_select.value
    if comp == 'Hx':
        y_fdtd, y_cal = np.abs(hx_f[fidx]), np.abs(hx_cal[fidx])
    else:
        y_fdtd, y_cal = np.abs(hz_f[fidx]), np.abs(hz_cal[fidx])
    geo = fdtd['geometry']
    rx_local = np.asarray(geo.get('rx_local_idx_per_trace', np.arange(y_fdtd.size)), dtype=int)
    order = np.argsort(rx_local)
    x = rx_local[order]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x, y=y_fdtd[order], mode='lines+markers', name='FDTD'))
    fig.add_trace(go.Scatter(x=x, y=y_cal[order], mode='lines+markers', name='C × analytic'))
    fig.update_layout(
        title=f'Calibration QC: {comp} amplitude vs rx (f={freqs[fidx]:g} Hz, {cal.get("method", "?")})',
        xaxis_title='Local rx index', yaxis_title='Amplitude', height=420,
    )
    with cal_plot_out:
        cal_plot_out.clear_output(wait=True)
        fig.show()


nproc_input = ipw.BoundedIntText(
    value=min(CONFIG.nproc_default, MAX_CPUS), min=2, max=MAX_CPUS,
    description='nproc', layout=ipw.Layout(width='200px'),
)
stop_button = ipw.Button(description='Stop run', button_style='warning')
refresh_button = ipw.Button(description='Refresh run status')
run_status = ipw.HTML(value='No active run process in this session.')
job_progress_out = ipw.Textarea(value='', description='job progress', layout=ipw.Layout(width='980px', height='180px'))
mpiqueue_out = ipw.Textarea(value='', description='mpiqueue', layout=ipw.Layout(width='980px', height='180px'))
worker_logs_out = ipw.Textarea(value='', description='worker logs', layout=ipw.Layout(width='980px', height='180px'))

load_button = ipw.Button(description='Load FD outputs')
load_info = ipw.HTML(value='Outputs not loaded yet.')

# VIEW selectors. These choose what the PLOTS below show - never what runs.
# Every action in this notebook (modelling, gain extraction, calibration,
# saving) is on ALL datasets; see RULE 2 in the project skill.
_ds_all = available_datasets()
view_dataset = ipw.Dropdown(
    options=([(d['name'], d['name']) for d in _ds_all] or [('(none built)', '')]),
    value=(_ds_all[0]['name'] if _ds_all else ''),
    description='view dataset', layout=ipw.Layout(width='420px'),
    style={'description_width': '100px'},
)
dataset_info = ipw.HTML(value=(
    f"{len(_ds_all)} dataset(s) in the acquisition matrix. Actions run on ALL of "
    "them; this selector only chooses which one is plotted." if _ds_all else
    "No datasets found - run Step 01 first."))


def on_view_dataset_change(_=None):
    ds = available_datasets()
    chosen = next((d for d in ds if d['name'] == view_dataset.value), None)
    if chosen is None:
        return
    m = chosen.get('meta', {})
    dataset_info.value = (
        f"<code>{chosen['name']}</code>: source={m.get('source_field', '?')}, "
        f"flist={m.get('flist_hz')}, dx={m.get('dx_model_target_m')} m, "
        f"order={m.get('fd_order')}"
    )
    by_ds = state.get('results_by_dataset') or {}
    if view_dataset.value in by_ds:
        state['result'] = by_ds[view_dataset.value]
        _sync_view_freqs()
        update_plot()


def _sync_view_freqs():
    res = state.get('result')
    if res is None:
        return
    freqs = np.asarray(res['Hx']['freqs'], dtype=float)
    comp_freq.options = [(f'{f:g} Hz', i) for i, f in enumerate(freqs)]
    comp_freq.value = 0


view_dataset.observe(lambda _c: on_view_dataset_change(), names='value')
_select_dataset(None)

run_all_btn = ipw.Button(description='Run modelling for ALL datasets (sequential)',
                         button_style='warning', layout=ipw.Layout(width='340px'))


def on_run_all(_):
    """Model every dataset of the matrix, one after another.

    Sequential on purpose: the datasets are independent so they COULD run
    concurrently, but each already uses every core through mpirun, so
    overlapping them on one machine only adds contention.

    Three things this has to get right, all of which it used to get wrong:

    * RE-ENTRANCY. The guard has to be set SYNCHRONOUSLY, before the worker
      thread starts. Checking `state['process']` did nothing, because the
      worker never set it - so every click launched another batch on top of
      the last, all writing into the same directories.
    * PROGRESS. `state['run_started']` and `state['active_run_dir']` have to be
      set, or `refresh_run_status` returns early and the per-job panel stays on
      "Run not started in this session yet." for the whole run.
    * THE PROCESS HANDLE. `subprocess.run` blocks and returns nothing to track,
      so Stop had nothing to kill and the status line never showed a pid.
    """
    if state.get('batch_running'):
        push_message('A batch is already running in this session - ignoring the click.')
        return
    ds = available_datasets()
    if not ds:
        push_message('No datasets to run - run Step 01 first.')
        return

    # Claim the batch BEFORE the thread exists, so a second click cannot slip in.
    state['batch_running'] = True
    state['batch_stop'] = False
    state['run_started'] = True
    run_all_btn.disabled = True
    run_all_btn.description = 'Running all datasets...'
    job_progress_out.value = 'Waiting for job logs...'
    worker_logs_out.value = 'Waiting for worker logs...'
    mpiqueue_out.value = 'Waiting for mpiqueue.log...'

    nproc = max(2, min(int(getattr(nproc_input, 'value', 4)), MAX_CPUS))
    engine = str(rockem_bridge.binary_path(CONFIG.forward_engine_te2d()))

    def _worker():
        failures = []
        try:
            for i, d in enumerate(ds, 1):
                if state.get('batch_stop'):
                    push_message(f'Batch stopped before {d["name"]}.')
                    break
                rd = Path(d['run_dir'])
                cfg_name = str((d.get('meta') or {}).get('forward_cfg', 'mod.cfg'))
                state['batch'] = {'i': i, 'n': len(ds), 'name': d['name']}
                state['active_run_dir'] = str(rd)
                (rd / 'Data').mkdir(parents=True, exist_ok=True)
                clear_stale_run_logs(run_dir=rd)
                push_message(f'[{i}/{len(ds)}] running {d["name"]} ({cfg_name}, -np {nproc}) ...')
                t0 = time.time()
                try:
                    proc = subprocess.Popen(
                        [CONFIG.mpirun, '-np', str(nproc), engine, cfg_name],
                        cwd=str(rd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True,
                    )
                    state['process'] = proc
                    out, _ = proc.communicate()
                    rc = proc.returncode
                except Exception as exc:
                    failures.append(f'{d["name"]}: {type(exc).__name__}: {exc}')
                    push_message(f'[{i}/{len(ds)}] {d["name"]} FAILED to start: {exc}')
                    continue
                finally:
                    state['process'] = None
                if rc != 0:
                    failures.append(f'{d["name"]} (rc={rc})')
                    push_message(f'[{i}/{len(ds)}] {d["name"]} FAILED rc={rc}: '
                                 + (out or '')[-400:])
                    # One bad dataset must not abandon the rest of a batch that
                    # may already be an hour in.
                    continue
                push_message(f'[{i}/{len(ds)}] {d["name"]} done in {time.time() - t0:.1f} s')
            if failures:
                push_message(f'Batch finished with {len(failures)} failure(s): '
                             + '; '.join(failures))
            else:
                push_message(f'All {len(ds)} dataset(s) modelled.')
        finally:
            state['batch_running'] = False
            state['batch'] = None
            state['process'] = None
            run_all_btn.disabled = False
            run_all_btn.description = 'Run modelling for ALL datasets (sequential)'
            try:
                refresh_run_status()
            except Exception:
                pass
            stop_auto_refresh()

    threading.Thread(target=_worker, daemon=True).start()
    refresh_run_status()
    start_auto_refresh()


run_all_btn.on_click(on_run_all)   # not bind_button_with_feedback: that reports
                                   # 'completed successfully' the moment the
                                   # handler returns, which for a background
                                   # batch is before anything has run.

compute_button = ipw.Button(description='Compute channel-gain amplitude/phase', button_style='primary')
compute_info = ipw.HTML(value='Not computed yet.')

cal_info = ipw.HTML(value='Calibration not computed yet.')
# Magnetic SOURCE component. The workshop used to hardcode a Kx (source_type=3)
# line source everywhere; Kz (source_type=5) is the transverse dipole, and the
# two runs together complete the 2x2 magnetic coupling matrix, from which any
# tilted transmitter/receiver pair follows by exact superposition.
calibrate_all_btn = ipw.Button(
    description='Calibrate ALL datasets (each with its own source)', button_style='warning',
    layout=ipw.Layout(width='340px'))
cal_all_method = ipw.Dropdown(
    options=[('1D lateral average of true model', METHOD_LATERAL_AVERAGE),
             ('Homogeneous rho_min', METHOD_HOMOGENEOUS)],
    value=METHOD_LATERAL_AVERAGE, description='batch method',
    layout=ipw.Layout(width='420px'), style={'description_width': '110px'})

cal_plot_out = ipw.Output(layout=ipw.Layout(width='980px', border='1px solid #ddd'))

component_select = ipw.Dropdown(options=[('Hx', 'Hx'), ('Hz', 'Hz')], value='Hx', description='component')
comp_freq = ipw.Dropdown(options=[('n/a', 0)], value=0, description='frequency')
cal_comp_select = ipw.Dropdown(options=[('Hx', 'Hx'), ('Hz', 'Hz')], value='Hx', description='cal comp')
cal_freq_select = ipw.Dropdown(options=[('n/a', 0)], value=0, description='cal freq')
metric_select = ipw.Dropdown(
    options=[
        ('Amplitude vs rx (local index)', 'amp_vs_rx'),
        ('Phase vs rx (deg, local index)', 'phase_vs_rx_deg'),
        ('Amplitude vs tx (fixed local rx)', 'amp_vs_tx'),
        ('Phase vs tx (deg, fixed local rx)', 'phase_vs_tx_deg'),
        ('Amplitude vs frequency', 'amp_vs_freq'),
        ('Phase vs frequency (deg)', 'phase_vs_freq_deg'),
    ],
    value='amp_vs_rx',
    description='plot',
)
tx_select = ipw.Dropdown(options=[('n/a', 0)], value=0, description='tx')
rx_local_select = ipw.IntSlider(value=0, min=0, max=0, step=1, description='local rx', continuous_update=False, layout=ipw.Layout(width='320px'))
trace_idx = ipw.IntSlider(value=0, min=0, max=0, step=1, description='trace idx', continuous_update=False, layout=ipw.Layout(width='420px'))
save_button = ipw.Button(description='Save processed results', button_style='info')
quit_button = ipw.Button(description='Quit GUI server', button_style='danger')
plot_out = ipw.Output(layout=ipw.Layout(width='980px', border='1px solid #ddd'))

status_area = ipw.Textarea(value='Ready.', description='Status', layout=ipw.Layout(width='980px', height='180px'))

bind_button_with_feedback(stop_button, on_stop_model, 'Stopping modelling run')
bind_button_with_feedback(refresh_button, refresh_run_status, 'Refreshing run status')
bind_button_with_feedback(load_button, on_load_outputs, 'Loading FD outputs')
bind_button_with_feedback(compute_button, on_compute, 'Computing amplitude/phase')
bind_button_with_feedback(calibrate_all_btn, on_calibrate_all, 'Calibrating every dataset')
bind_button_with_feedback(save_button, on_save_results, 'Saving processed results')
bind_button_with_feedback(quit_button, on_quit_gui, 'Shutting down visualization GUI server')
component_select.observe(update_plot, names='value')
comp_freq.observe(update_plot, names='value')
cal_comp_select.observe(update_calibration_plot, names='value')
cal_freq_select.observe(update_calibration_plot, names='value')
metric_select.observe(update_plot, names='value')
tx_select.observe(update_plot, names='value')
rx_local_select.observe(update_plot, names='value')
trace_idx.observe(update_plot, names='value')


from IPython.display import display

app_header = ipw.HTML(
    '<h2>02_fwmodelling_and_data_visualization</h2>'
    '<p>Run FD modelling (explicit TE2D engine) and extract steady-state '
    'channel-gain amplitude/phase from Hx/Hz shot records, referenced to the '
    'injected wavelet.</p>'
)

run_section = ipw.VBox([
    ipw.HTML('<h3>1) Run modelling</h3>'),
    ipw.HTML(
        '<p>Step 01 built one dataset per (frequency, source component) pair. '
        'Every action here runs <b>all</b> of them, in sequence. The '
        '<code>dataset</code> dropdown only chooses which one the plots below '
        'display - it never changes what runs.</p>'
    ),
    dataset_info,
    ipw.HBox([nproc_input, run_all_btn, stop_button, refresh_button]),
    run_status,
    ipw.HTML('<b>Per-job completion (auto-refresh every 2s while running)</b>'),
    job_progress_out,
    mpiqueue_out,
    worker_logs_out,
])

load_section = ipw.VBox([
    ipw.HTML('<h3>2) Load outputs</h3>'),
    ipw.HBox([load_button]),
    load_info,
])

compute_section = ipw.VBox([
    ipw.HTML(
        '<h3>3) Compute channel-gain amplitude/phase</h3>'
        '<p>Steady-state phasor ratio (trace / injected wavelet), for <b>every '
        'dataset</b> - see <code>scripts.modules.fd_visualization.steady_state_gains</code>. '
        'Each dataset is extracted at its OWN tone, <code>f_min</code> and '
        '<code>n_periods_extract</code>, read from its own '
        '<code>setup_metadata.json</code>. Step 01 wrote those; there is nothing '
        'to re-enter here.</p>'
    ),
    compute_button,
    compute_info,
])

calibration_section = ipw.VBox([
    ipw.HTML(
        '<h3>3b) FDTD–analytic calibration</h3>'
        '<p>Two options write a purpose-sized source-centred domain, run one explicit TE2D job, '
        'and fit a global complex <code>C(f)</code> (FDTD ≈ C × analytic; rockem-suite: |C| ≈ dx²). '
        'The <b>last successful</b> calibration overwrites <code>setup_metadata.json</code> for '
        'notebooks 05/06.</p>'
        '<ul>'
        '<li><b>Homogeneous rho_min</b> — Earth = <code>rho_min</code>; receivers at ±depth so Hz '
        'enters <code>C</code> even for colinear surveys.</li>'
        '<li><b>1D lateral average of true model</b> — Earth = mean resistivity across x of '
        'production <code>sg.rss</code> (vertically varying 1D). Survey offsets, <code>apertx</code>, '
        '<code>dx</code>/<code>dt</code>/<code>eps_r</code>/PML match Step 01. With layer contrasts, '
        'Hz need not vanish when <code>rz0=tz0</code>.</li>'
        '</ul>'
        '<p><b>cal source</b> selects the magnetic source component: <b>Kx</b> (<code>source_type=3</code>, the historical coaxial source, analytic counterpart <code>magnetic_line_source_fields_layered</code>) or <b>Kz</b> (<code>source_type=5</code>, transverse, <code>magnetic_z_line_source_fields_layered</code>). Each gets its own run directory and its own fitted <code>C(f)</code>, both stored under <code>fdtd_analytic_calibration_by_source</code>. Only a <b>Kx</b> calibration becomes the ACTIVE <code>C</code> for notebooks 05/06, because the 1D inversion forward model is a Kx line source. If Kx and Kz need different <code>C</code>, that is a source-normalisation bug, not physics: both inject through the same <code>dt/MU</code> coefficient into one cell, so both should give <code>|C| &approx; dx&sup2;</code>.</p><p>Running both completes the 2&times;2 magnetic coupling matrix, from which any tilted transmitter/receiver pair follows by exact superposition - see <code>scripts/experiments/fault_couplings.py</code>.</p><p>Runs block the GUI; domains are small so they finish quickly.</p>'
    ),
    ipw.HBox([calibrate_all_btn, cal_all_method]),
    cal_info,
    ipw.HBox([cal_comp_select, cal_freq_select]),
    cal_plot_out,
])

plot_section = ipw.VBox([
    ipw.HTML('<h3>4) Visualization and export</h3>'),
    ipw.HBox([view_dataset, component_select, metric_select, comp_freq, tx_select, rx_local_select]),
    trace_idx,
    ipw.HBox([save_button, quit_button]),
    ipw.HTML('<b>Session control</b>: use Quit GUI server to stop Voila even if browser is closed.'),
    plot_out,
])

gui = ipw.VBox([
    app_header,
    run_section,
    load_section,
    compute_section,
    calibration_section,
    plot_section,
    status_area,
])

clear_stale_run_logs(announce=False)
_meta_note = status_note_if_meta_missing(path=SETUP_META)
if _meta_note:
    push_message(_meta_note)
refresh_run_status()
display(gui)
